# Exploratory Analysis

**Master's Thesis – LLM-Specific Labor Market Exposure in Switzerland**

This notebook provides an interactive exploratory analysis of the processed data
and the computed exposure index.  It is intended as a sandbox for visualisation
and hypothesis generation ahead of the formal statistical analysis in
`scripts/05_statistical_analysis.R`.

## Contents
1. Load processed data
2. Inspect O\*NET combined table
3. Inspect CH-ISCO mapping
4. Exposure index overview
5. Interactive exploration

In [ ]:
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

PROCESSED = pathlib.Path('../data/processed')
OUTPUT    = pathlib.Path('../data/output')

## 1. Load Processed Data

In [ ]:
onet   = pd.read_parquet(PROCESSED / 'onet_combined.parquet')
ch     = pd.read_parquet(PROCESSED / 'ch_isco19_clean.parquet')
mapped = pd.read_parquet(PROCESSED / 'onet_ch_mapped.parquet')
mu     = pd.read_parquet(PROCESSED / 'mu_weights.parquet')

print('O*NET combined :', onet.shape)
print('CH-ISCO clean  :', ch.shape)
print('O*NET mapped   :', mapped.shape)
print('μ weights      :', mu.shape)

## 2. O\*NET Combined Table

In [ ]:
onet.head()

In [ ]:
# Category distribution
onet['onet_category'].value_counts().plot(kind='barh', title='O*NET rows by category')
plt.tight_layout()
plt.show()

## 3. CH-ISCO Mapping Quality

In [ ]:
ch.head()

In [ ]:
# Check mapping coverage
n_total   = len(mapped)
n_matched = mapped['isco_code'].notna().sum()
print(f'Mapping coverage: {n_matched}/{n_total} ({n_matched/n_total:.1%})')

## 4. Exposure Index Overview

In [ ]:
index_path = OUTPUT / 'exposure_index.parquet'
if index_path.exists():
    exposure = pd.read_parquet(index_path)
    print(exposure.shape)
    exposure.head()
else:
    print('Exposure index not yet computed – run script 04 first.')

In [ ]:
if 'exposure' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    axes[0].hist(exposure['exposure_index'].dropna(), bins=20, color='steelblue', edgecolor='white')
    axes[0].set_title('Exposure Index Distribution')
    axes[0].set_xlabel('Exposure Index')
    axes[0].set_ylabel('Count')

    # Box-plot by ISCO major group
    exposure['isco1'] = exposure['isco4'].astype(str).str[0]
    exposure.boxplot(column='exposure_index', by='isco1', ax=axes[1])
    axes[1].set_title('Exposure by ISCO Major Group')
    axes[1].set_xlabel('ISCO Major Group')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()

## 5. μ Weights Inspection

In [ ]:
mu.groupby('onet_category')['mu'].describe().round(3)

In [ ]:
# Top elements by μ weight
mu.nlargest(20, 'mu')[['onet_category', 'element_id', 'mu']]